<a href="https://www.kaggle.com/code/lukaspanos/spaceship-titanic-with-pipeline?scriptVersionId=344266426" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier

In [2]:
spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

def engineer(df):
    df = df.copy()   # never mutate input → cell is rerunnable forever

    # --- Cabin ---
    df['Cabin'] = df['Cabin'].fillna('U/0/U')
    df[['Deck', 'CabinNum', 'Side']] = df['Cabin'].str.split('/', expand=True)
    df['CabinNum'] = df['CabinNum'].astype(float)

    # --- Group (from PassengerId) ---
    group = df['PassengerId'].str.split('_').str[0]
    df['GroupSize'] = group.map(group.value_counts())
    df['IsAloneGroup'] = (df['GroupSize'] == 1).astype(int)

    # --- Spending: logic fills only (stateless rules) ---
    for col in spend_cols:
        df.loc[(df['CryoSleep'] == True) & (df[col].isnull()), col] = 0
    df['TotalSpend'] = df[spend_cols].sum(axis=1)

    # --- CryoSleep logic fill ---
    df.loc[df['CryoSleep'].isnull() & (df['TotalSpend'] == 0), 'CryoSleep'] = True
    df.loc[df['CryoSleep'].isnull() & (df['TotalSpend'] > 0), 'CryoSleep'] = False

    # --- Flags ---
    df['NoSpend'] = (df['TotalSpend'] == 0).astype(int)
    df['AwakeNoSpend'] = ((df['CryoSleep'] == False) & (df['NoSpend'] == 1)).astype(int)

    # --- Skew fix (still-NaN spend stays NaN; imputer handles it later, in log space) ---
    for col in spend_cols + ['TotalSpend']:
        df[col] = np.log1p(df[col])

    # --- Bools to numeric, NaN preserved for the imputer ---
    df['CryoSleep'] = df['CryoSleep'].astype(float)
    df['VIP'] = df['VIP'].astype(float)

    # --- Drop what the model never sees ---
    return df.drop(columns=['PassengerId', 'Name', 'Cabin'])

In [3]:
train_raw = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/train.csv')
test_raw  = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/test.csv')

test_ids = test_raw['PassengerId']

X = engineer(train_raw.drop(columns=['Transported']))
y = train_raw['Transported'].astype(int)
X_test_final = engineer(test_raw)

In [4]:
numeric_features = ['Age', 'CabinNum', 'GroupSize', 'IsAloneGroup',
                    'CryoSleep', 'VIP', 'NoSpend', 'AwakeNoSpend',
                    'TotalSpend'] + spend_cols

categorical_features = ['HomePlanet', 'Destination', 'Deck', 'Side']

preprocessor = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), numeric_features),
    ('cat', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), categorical_features)
])

model = Pipeline([
    ('prep', preprocessor),
    ('xgb', XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=3,
                          subsample=1.0, colsample_bytree=0.7, min_child_weight=1,
                          random_state=42))
])

In [5]:
scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
print(scores)
print(f"CV: {scores.mean():.4f} ± {scores.std():.4f}")

[0.77228292 0.79010926 0.80506038 0.82451093 0.8009206 ]
CV: 0.7986 ± 0.0172


In [6]:
model.fit(X, y)
preds = model.predict(X_test_final)

submission = pd.DataFrame({'PassengerId': test_ids,
                           'Transported': preds.astype(bool)})
submission.to_csv('submission.csv', index=False)
print(submission.head())

  PassengerId  Transported
0     0013_01         True
1     0018_01        False
2     0019_01         True
3     0021_01         True
4     0023_01         True
